# When the Church Votes Left: How Progressive Bishops Supported the Workers' Party in Brazil

**Authors:** Tuñón, Guadalupe

**Journal:** American Political Science Review (2026)

This notebook reproduces the analysis from the paper above.
It was auto-generated by [PoliRep](https://polirep.org).

---

## Setup

Install packages and import standard libraries.

In [ ]:
!pip install -q pandas numpy matplotlib scipy statsmodels pyfixest

import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

## Source Modules

Write the paper's `src/` package to disk so `from src.* import ...` works.

In [ ]:
import sys
from pathlib import Path

# Write src/ package to Colab filesystem
Path("src").mkdir(exist_ok=True)
Path("src/__init__.py").write_text("")
Path("src/analysis_electoral.py").write_text("\"\"\"Electoral analysis: Tables 1, C1-C8, D1.\n\nMain results on PT presidential vote share.\n\"\"\"\n\nimport pandas as pd\nimport pyfixest as pf\n\nfrom .config import ELECTION_YEARS\nfrom .data import load_main_data, prepare_analysis_sample\nfrom .helpers import extract_pyfixest_results, save_table_latex\n\n\ndef table1(sample: pd.DataFrame = None) -> pd.DataFrame:\n    \"\"\"Table 1: Main results - PT presidential vote share (2SLS and ITT).\n\n    Args:\n        sample: Optional pre-prepared sample. If None, loads and prepares data.\n\n    Returns:\n        DataFrame with regression results\n    \"\"\"\n    if sample is None:\n        df = load_main_data()\n        sample = prepare_analysis_sample(df, exclude_archdioceses=True)\n\n    results_2sls = []\n    results_itt = []\n    outcome_means = []\n\n    for year in ELECTION_YEARS:\n        # Filter to specific election year\n        df_year = sample[sample[\"election\"] == year].copy()\n\n        # Compute outcome mean\n        outcome_means.append(df_year[\"vote.sh\"].mean())\n\n        # Panel A: 2SLS (CACE)\n        # vote.sh ~ Exposure | Retirement_Years + UF | cluster(CE_1978)\n        fit_2sls = pf.feols(\n            \"vote.sh ~ 1 | UF | Exposure ~ Retirement_Years\",\n            data=df_year,\n            vcov={\"CRV1\": \"CE_1978\"},\n        )\n        results_2sls.append(extract_pyfixest_results(fit_2sls, \"Exposure\"))\n\n        # Panel B: ITT (Reduced Form)\n        # vote.sh ~ Retirement_Years + UF | cluster(CE_1978)\n        fit_itt = pf.feols(\n            \"vote.sh ~ Retirement_Years | UF\",\n            data=df_year,\n            vcov={\"CRV1\": \"CE_1978\"},\n        )\n        results_itt.append(extract_pyfixest_results(fit_itt, \"Retirement_Years\"))\n\n    # Build table\n    table = build_table1(results_2sls, results_itt, outcome_means, ELECTION_YEARS)\n\n    # Save\n    save_table_latex(\n        table,\n        \"table1.tex\",\n        caption=\"PT Presidential Vote Share (Main Results)\",\n        label=\"tab:main\",\n    )\n\n    return table\n\n\ndef build_table1(\n    results_2sls: list, results_itt: list, outcome_means: list, years: list\n) -> pd.DataFrame:\n    \"\"\"Build Table 1 with Panel A (2SLS) and Panel B (ITT).\n\n    Args:\n        results_2sls: List of 2SLS results for each year\n        results_itt: List of ITT results for each year\n        outcome_means: List of outcome means for each year\n        years: List of election years\n\n    Returns:\n        Formatted DataFrame\n    \"\"\"\n    rows = []\n\n    # Header row\n    rows.append([\"\"] + [str(y) for y in years])\n\n    # Panel A: 2SLS\n    rows.append([\"\\\\textbf{Panel A: 2SLS (CACE)}\"] + [\"\"] * len(years))\n    rows.append([\"Exposure\"] + [\n        f\"{r['coef']:.3f}{get_stars(r['pval'])}\" for r in results_2sls\n    ])\n    rows.append([\"\"] + [f\"({r['se']:.3f})\" for r in results_2sls])\n\n    # Panel B: ITT\n    rows.append([\"\\\\textbf{Panel B: ITT (Reduced Form)}\"] + [\"\"] * len(years))\n    rows.append([\"Retirement Years\"] + [\n        f\"{r['coef']:.3f}{get_stars(r['pval'])}\" for r in results_itt\n    ])\n    rows.append([\"\"] + [f\"({r['se']:.3f})\" for r in results_itt])\n\n    # Metadata\n    rows.append([\"\"] + [\"\"] * len(years))\n    rows.append([\"Outcome mean\"] + [f\"{m:.2f}\" for m in outcome_means])\n    rows.append([\"N obs\"] + [str(r[\"nobs\"]) for r in results_2sls])\n    rows.append([\"State FE\"] + [\"Yes\"] * len(years))\n    rows.append([\"SE\"] + [\"Clustered (Diocese)\"] * len(years))\n\n    return pd.DataFrame(rows)\n\n\ndef tableC1(sample: pd.DataFrame = None) -> pd.DataFrame:\n    \"\"\"Table C1: Diocese-level analysis.\n\n    Aggregate municipality votes to diocese level.\n    \"\"\"\n    if sample is None:\n        df = load_main_data()\n        sample = prepare_analysis_sample(df, exclude_archdioceses=True)\n\n    # Aggregate to diocese level\n    from .helpers import aggregate_to_diocese\n\n    # Sum votes by diocese-election\n    agg_vars = [\"votos\", \"votos.sum\"]\n    diocese_data = aggregate_to_diocese(sample, agg_vars, diocese_var=\"CE_code\")\n\n    # Compute vote share at diocese level\n    diocese_data[\"vote.sh\"] = 100 * diocese_data[\"votos\"] / diocese_data[\"votos.sum\"]\n\n    # Merge treatment variables (already at diocese level)\n    diocese_data = diocese_data.merge(\n        sample[[\"CE_code\", \"election\", \"Exposure\", \"Retirement_Years\", \"UF_dio\"]].drop_duplicates(),\n        on=[\"CE_code\", \"election\"],\n        how=\"left\",\n    )\n\n    results_2sls = []\n    results_itt = []\n    outcome_means = []\n\n    for year in ELECTION_YEARS:\n        df_year = diocese_data[diocese_data[\"election\"] == year].copy()\n        outcome_means.append(df_year[\"vote.sh\"].mean())\n\n        # 2SLS with state FE (no clustering at diocese level)\n        fit_2sls = pf.feols(\n            \"vote.sh ~ 1 | UF_dio | Exposure ~ Retirement_Years\",\n            data=df_year,\n            vcov=\"iid\",  # No clustering for diocese-level\n        )\n        results_2sls.append(extract_pyfixest_results(fit_2sls, \"Exposure\"))\n\n        # ITT\n        fit_itt = pf.feols(\n            \"vote.sh ~ Retirement_Years | UF_dio\", data=df_year, vcov=\"iid\"\n        )\n        results_itt.append(extract_pyfixest_results(fit_itt, \"Retirement_Years\"))\n\n    # Build and save table\n    table = build_table1(results_2sls, results_itt, outcome_means, ELECTION_YEARS)\n    save_table_latex(\n        table, \"tableC1.tex\", caption=\"Diocese-Level Analysis\", label=\"tab:diocese\"\n    )\n\n    return table\n\n\ndef tableC2(sample: pd.DataFrame = None) -> pd.DataFrame:\n    \"\"\"Table C2: Include archdioceses.\"\"\"\n    if sample is None:\n        df = load_main_data()\n        # Include archdioceses this time\n        sample = prepare_analysis_sample(df, exclude_archdioceses=False)\n\n    return table1(sample)\n\n\ndef tableC5(sample: pd.DataFrame = None) -> pd.DataFrame:\n    \"\"\"Table C5: Truncated sample (I_YEAR < election only).\"\"\"\n    if sample is None:\n        df = load_main_data()\n        sample = prepare_analysis_sample(df, exclude_archdioceses=True)\n\n    # Truncate: only include dioceses where retirement occurred before election\n    sample = sample[sample[\"I_YEAR\"] < sample[\"election\"]].copy()\n\n    return table1(sample)\n\n\ndef tableD1(sample: pd.DataFrame = None) -> pd.DataFrame:\n    \"\"\"Table D1: All left-wing parties (PT, PDT, PSB, PPS, PCDOB).\"\"\"\n    if sample is None:\n        df = load_main_data()\n\n        # Load and merge electoral data for all left parties\n        df = df[(df[\"partido\"].isin([\"PT\", \"PDT\", \"PSB\", \"PPS\", \"PCDOB\"]))]\n        df = df[df[\"cargo\"] == \"Presidente\"]\n        df = df[df[\"CE_TYPE\"] != \"A\"]\n\n        # Construct treatment variables\n        from .data import construct_treatment_variables\n\n        sample = construct_treatment_variables(df)\n\n    # Group by municipality-election and sum votes across left parties\n    left_agg = (\n        sample.groupby([\"cod.2010\", \"election\", \"CE_1978\", \"UF\", \"Retirement_Years\", \"Exposure\"])\n        .agg({\"votos\": \"sum\", \"votos.uteis\": \"first\"})\n        .reset_index()\n    )\n    left_agg[\"vote.sh\"] = 100 * left_agg[\"votos\"] / left_agg[\"votos.uteis\"]\n\n    return table1(left_agg)\n\n\ndef get_stars(pval: float) -> str:\n    \"\"\"Get significance stars from p-value.\"\"\"\n    if pval < 0.01:\n        return \"***\"\n    elif pval < 0.05:\n        return \"**\"\n    elif pval < 0.10:\n        return \"*\"\n    return \"\"\n\n\ndef run(sample: pd.DataFrame = None):\n    \"\"\"Run all electoral analyses.\"\"\"\n    print(\"Running Table 1 (Main Results)...\")\n    table1(sample)\n\n    print(\"Running Table C1 (Diocese-Level)...\")\n    tableC1(sample)\n\n    print(\"Running Table C2 (Include Archdioceses)...\")\n    tableC2(sample)\n\n    print(\"Running Table C5 (Truncated Sample)...\")\n    tableC5(sample)\n\n    print(\"Running Table D1 (All Left Parties)...\")\n    tableD1(sample)\n\n    print(\"Electoral analysis complete.\")\n")
Path("src/analysis_localPT.py").write_text("\"\"\"Local PT analysis: Table 3 (PT Office Activation and New Members.\"\"\"\n\nimport pandas as pd\nimport pyfixest as pf\n\nfrom .data import load_pt_panel_data, prepare_pt_panel_sample\nfrom .helpers import extract_pyfixest_results, save_table_latex\n\n\ndef table3(sample: pd.DataFrame = None) -> pd.DataFrame:\n    \"\"\"Table 3: PT local presence (2SLS and ITT).\n\n    2 columns:\n    1: had_office (binary indicator)\n    2: new_members (count)\n\n    Args:\n        sample: Optional pre-prepared sample. If None, loads and prepares data.\n\n    Returns:\n        DataFrame with regression results\n    \"\"\"\n    if sample is None:\n        df = load_pt_panel_data()\n        sample = prepare_pt_panel_sample(df)\n\n    outcomes = [\"had_office\", \"new_members\"]\n    results_2sls = []\n    results_itt = []\n    outcome_means = []\n\n    for outcome in outcomes:\n        # Compute outcome mean\n        outcome_means.append(sample[outcome].mean())\n\n        # Panel A: 2SLS\n        # outcome ~ cons | I_cons + cod.1970 + year | cluster(CE_1978)\n        fit_2sls = pf.feols(\n            f\"{outcome} ~ 1 | cod.1970 + year | cons ~ I_cons\",\n            data=sample,\n            vcov={\"CRV1\": \"CE_1978\"},\n        )\n        results_2sls.append(extract_pyfixest_results(fit_2sls, \"cons\"))\n\n        # Panel B: ITT\n        # outcome ~ I_cons + cod.1970 + year | cluster(CE_1978)\n        fit_itt = pf.feols(\n            f\"{outcome} ~ I_cons | cod.1970 + year\",\n            data=sample,\n            vcov={\"CRV1\": \"CE_1978\"},\n        )\n        results_itt.append(extract_pyfixest_results(fit_itt, \"I_cons\"))\n\n    # Build table\n    table = build_table3(results_2sls, results_itt, outcome_means)\n\n    # Save\n    save_table_latex(\n        table,\n        \"table3.tex\",\n        caption=\"PT Local Office Activation and New Members\",\n        label=\"tab:local_pt\",\n    )\n\n    return table\n\n\ndef build_table3(results_2sls: list, results_itt: list, outcome_means: list) -> pd.DataFrame:\n    \"\"\"Build Table 3 with 2 columns.\"\"\"\n    rows = []\n\n    # Header\n    col_labels = [\"Had Office\", \"New Members\"]\n    rows.append([\"\"] + col_labels)\n\n    # Panel A: 2SLS\n    rows.append([\"\\\\textbf{Panel A: 2SLS}\"] + [\"\"] * 2)\n    rows.append([\"JPII Bishop\"] + [\n        f\"{r['coef']:.3f}{get_stars(r['pval'])}\" for r in results_2sls\n    ])\n    rows.append([\"\"] + [f\"({r['se']:.3f})\" for r in results_2sls])\n\n    # Panel B: ITT\n    rows.append([\"\\\\textbf{Panel B: ITT}\"] + [\"\"] * 2)\n    rows.append([\"Mandated JPII Bishop\"] + [\n        f\"{r['coef']:.3f}{get_stars(r['pval'])}\" for r in results_itt\n    ])\n    rows.append([\"\"] + [f\"({r['se']:.3f})\" for r in results_itt])\n\n    # Metadata\n    rows.append([\"\"] + [\"\"] * 2)\n    rows.append([\"Outcome mean\"] + [f\"{m:.3f}\" for m in outcome_means])\n    rows.append([\"N obs\"] + [str(r[\"nobs\"]) for r in results_2sls])\n    rows.append([\"Municipality FE\"] + [\"Yes\"] * 2)\n    rows.append([\"Year FE\"] + [\"Yes\"] * 2)\n    rows.append([\"SE\"] + [\"Clustered (Diocese)\"] * 2)\n\n    return pd.DataFrame(rows)\n\n\ndef get_stars(pval: float) -> str:\n    \"\"\"Get significance stars from p-value.\"\"\"\n    if pval < 0.01:\n        return \"***\"\n    elif pval < 0.05:\n        return \"**\"\n    elif pval < 0.10:\n        return \"*\"\n    return \"\"\n\n\ndef run(sample: pd.DataFrame = None):\n    \"\"\"Run PT local presence analysis.\"\"\"\n    print(\"Running Table 3 (PT Local Presence)...\")\n    table3(sample)\n    print(\"PT local presence analysis complete.\")\n")
Path("src/analysis_parishdata.py").write_text("\"\"\"Parish data analysis: Table 2 (Priest Turnover.\"\"\"\n\nimport pandas as pd\nimport pyfixest as pf\n\nfrom .data import load_parish_data, prepare_parish_sample\nfrom .helpers import extract_pyfixest_results, save_table_latex\n\n\ndef table2(sample: pd.DataFrame = None) -> pd.DataFrame:\n    \"\"\"Table 2: Priest turnover (2SLS and ITT).\n\n    6 columns:\n    1-2: All parishes (binary and share outcomes)\n    3: Single-parish municipalities\n    4: Single secular parishes\n    5-6: RS state progressives vs non-progressives\n\n    Args:\n        sample: Optional pre-prepared sample. If None, loads and prepares data.\n\n    Returns:\n        DataFrame with regression results\n    \"\"\"\n    if sample is None:\n        df = load_parish_data()\n        sample = prepare_parish_sample(df)\n\n    # Column specifications\n    specs = [\n        # Col 1: All parishes, binary turnover\n        {\"filter\": None, \"outcome\": \"turnover_priest_01\"},\n        # Col 2: All parishes, share turnover\n        {\"filter\": None, \"outcome\": \"turnover_priest_sh\"},\n        # Col 3: Single parish municipalities\n        {\"filter\": lambda df: df[\"parishes_1977\"] == 1, \"outcome\": \"turnover_priest_01\"},\n        # Col 4: Single secular parishes\n        {\n            \"filter\": lambda df: (df[\"parishes_1977\"] == 1) & (df[\"parishes_1977_order\"] == 0),\n            \"outcome\": \"turnover_priest_01\",\n        },\n        # Col 5: RS progressives\n        {\n            \"filter\": lambda df: (df[\"UF\"] == \"RS\") & (df[\"AC_year\"] > 1970) & (df[\"prog_priest_77\"] == 1),\n            \"outcome\": \"turnover_priest_01\",\n        },\n        # Col 6: RS non-progressives\n        {\n            \"filter\": lambda df: (df[\"UF\"] == \"RS\") & (df[\"AC_year\"] > 1970) & (df[\"prog_priest_77\"] == 0),\n            \"outcome\": \"turnover_priest_01\",\n        },\n    ]\n\n    results_2sls = []\n    results_itt = []\n    outcome_means = []\n\n    for spec in specs:\n        # Apply filter if specified\n        if spec[\"filter\"] is not None:\n            df_spec = spec[\"filter\"](sample.copy())\n        else:\n            df_spec = sample.copy()\n\n        # Drop missing outcomes\n        df_spec = df_spec[df_spec[spec[\"outcome\"]].notna()]\n\n        # Compute outcome mean\n        outcome_means.append(df_spec[spec[\"outcome\"]].mean())\n\n        # Panel A: 2SLS\n        # outcome ~ cons | I_cons + cod.1970 + AC_year | cluster(CE_1978)\n        # Fixed effects: municipality + yearbook\n        fit_2sls = pf.feols(\n            f\"{spec['outcome']} ~ 1 | cod.1970 + AC_year | cons ~ I_cons\",\n            data=df_spec,\n            vcov={\"CRV1\": \"CE_1978\"},\n        )\n        results_2sls.append(extract_pyfixest_results(fit_2sls, \"cons\"))\n\n        # Panel B: ITT\n        # outcome ~ I_cons + cod.1970 + AC_year | cluster(CE_1978)\n        fit_itt = pf.feols(\n            f\"{spec['outcome']} ~ I_cons | cod.1970 + AC_year\",\n            data=df_spec,\n            vcov={\"CRV1\": \"CE_1978\"},\n        )\n        results_itt.append(extract_pyfixest_results(fit_itt, \"I_cons\"))\n\n    # Build table\n    table = build_table2(results_2sls, results_itt, outcome_means)\n\n    # Save\n    save_table_latex(\n        table, \"table2.tex\", caption=\"Priest Turnover\", label=\"tab:priest\"\n    )\n\n    return table\n\n\ndef build_table2(results_2sls: list, results_itt: list, outcome_means: list) -> pd.DataFrame:\n    \"\"\"Build Table 2 with 6 columns.\"\"\"\n    rows = []\n\n    # Header\n    col_labels = [\n        \"All (Binary)\",\n        \"All (Share)\",\n        \"Single Parish\",\n        \"Single Secular\",\n        \"RS Progressive\",\n        \"RS Non-Prog\",\n    ]\n    rows.append([\"\"] + col_labels)\n\n    # Panel A: 2SLS\n    rows.append([\"\\\\textbf{Panel A: 2SLS}\"] + [\"\"] * 6)\n    rows.append([\"JPII Bishop\"] + [\n        f\"{r['coef']:.3f}{get_stars(r['pval'])}\" for r in results_2sls\n    ])\n    rows.append([\"\"] + [f\"({r['se']:.3f})\" for r in results_2sls])\n\n    # Panel B: ITT\n    rows.append([\"\\\\textbf{Panel B: ITT}\"] + [\"\"] * 6)\n    rows.append([\"Mandated JPII Bishop\"] + [\n        f\"{r['coef']:.3f}{get_stars(r['pval'])}\" for r in results_itt\n    ])\n    rows.append([\"\"] + [f\"({r['se']:.3f})\" for r in results_itt])\n\n    # Metadata\n    rows.append([\"\"] + [\"\"] * 6)\n    rows.append([\"Outcome mean\"] + [f\"{m:.3f}\" for m in outcome_means])\n    rows.append([\"N obs\"] + [str(r[\"nobs\"]) for r in results_2sls])\n    rows.append([\"Municipality FE\"] + [\"Yes\"] * 6)\n    rows.append([\"Yearbook FE\"] + [\"Yes\"] * 6)\n    rows.append([\"SE\"] + [\"Clustered (Diocese)\"] * 6)\n\n    return pd.DataFrame(rows)\n\n\ndef get_stars(pval: float) -> str:\n    \"\"\"Get significance stars from p-value.\"\"\"\n    if pval < 0.01:\n        return \"***\"\n    elif pval < 0.05:\n        return \"**\"\n    elif pval < 0.10:\n        return \"*\"\n    return \"\"\n\n\ndef run(sample: pd.DataFrame = None):\n    \"\"\"Run parish data analysis.\"\"\"\n    print(\"Running Table 2 (Priest Turnover)...\")\n    table2(sample)\n    print(\"Parish data analysis complete.\")\n")
Path("src/config.py").write_text("\"\"\"Configuration: paths and constants for this paper's reproduction.\"\"\"\n\nfrom pathlib import Path\n\n# ---------------------------------------------------------------------------\n# Directory structure\n# ---------------------------------------------------------------------------\n\n# papers/<slug>/\nPAPER_ROOT = Path(\".\")\n\n# Top-level project root\nPROJECT_ROOT = PAPER_ROOT\n\n# Data paths\nDATA_RAW = Path(\"data/raw\")\nDATA_CONVERTED = Path(\"data/converted\")\n\n# Output paths\nOUTPUT_DIR = Path(\"output\")\nTABLE_DIR = OUTPUT_DIR / \"tables\"\nFIGURE_DIR = OUTPUT_DIR / \"figures\"\n\n# Original outputs for comparison\nORIGINAL_TABLE_DIR = Path(\"original/tables\")\nORIGINAL_FIGURE_DIR = Path(\"original/figures\")\n\n\n# ---------------------------------------------------------------------------\n# Analysis constants\n# ---------------------------------------------------------------------------\n\n# Election years\nELECTION_YEARS = [1989, 1994, 1998, 2002]\n\n# Sample restrictions\nEXCLUDE_ARCHDIOCESES = True  # CE_TYPE != \"A\"\n\n# Treatment/outcome variables\nTREATMENT_VAR = \"Exposure\"  # JPII bishop exposure\nINSTRUMENT_VAR = \"Retirement_Years\"  # Mandated exposure\nOUTCOME_VAR_ELECTORAL = \"vote.sh\"  # PT vote share\n\n# Key identifiers\nDIOCESE_ID = \"CE_1978\"  # Diocese (clustering variable)\nMUNI_ID_2010 = \"cod.2010\"  # Municipality code (2010)\nMUNI_ID_1970 = \"cod.1970\"  # Municipality code (1970)\nSTATE_ID = \"UF\"  # State (fixed effects)\n\n# Data files\nDATA_FILES = {\n    \"munidata\": \"munidata.parquet\",\n    \"diodata\": \"diodata.parquet\",\n    \"electoral\": \"IPEA_electoral_data.parquet\",\n    \"parish\": \"muni_parish_database.parquet\",\n    \"pt_panel\": \"panel_filiaco_mun_1970_pt.parquet\",\n    \"bishop_class\": \"NIS_Bishop-class.parquet\",\n    \"bishops_global\": \"bishops_all_countries.parquet\",\n}\n\n\n# ---------------------------------------------------------------------------\n# Helpers\n# ---------------------------------------------------------------------------\n\ndef ensure_output_dirs():\n    \"\"\"Create output directories if they don't exist.\"\"\"\n    TABLE_DIR.mkdir(parents=True, exist_ok=True)\n    FIGURE_DIR.mkdir(parents=True, exist_ok=True)\n")
Path("src/data.py").write_text("\"\"\"Data loading and sample construction.\n\nPortal compatibility requires these three functions:\n  - load_main_data() -> pd.DataFrame\n  - prepare_analysis_sample(df) -> pd.DataFrame\n  - get_sample_stats(sample) -> dict\n\"\"\"\n\nimport pandas as pd\n\ntry:\n    from .config import DATA_CONVERTED, DATA_FILES\nexcept ImportError:\n    # Fallback for direct script execution\n    from config import DATA_CONVERTED, DATA_FILES\n\n\ndef load_main_data() -> pd.DataFrame:\n    \"\"\"Load the main analysis dataset from converted data.\n\n    Returns:\n        DataFrame with all variables needed for analysis.\n    \"\"\"\n    # Load primary datasets\n    munidata = pd.read_parquet(DATA_CONVERTED / DATA_FILES[\"munidata\"])\n    electoral = pd.read_parquet(DATA_CONVERTED / DATA_FILES[\"electoral\"])\n\n    # Merge municipality data with electoral data\n    df = munidata.merge(electoral, on=\"cod.2010\", how=\"left\")\n\n    # Return merged dataset\n    return df\n\n\ndef load_diocese_data() -> pd.DataFrame:\n    \"\"\"Load diocese-level data.\n\n    Returns:\n        DataFrame with diocese characteristics.\n    \"\"\"\n    return pd.read_parquet(DATA_CONVERTED / DATA_FILES[\"diodata\"])\n\n\ndef load_parish_data() -> pd.DataFrame:\n    \"\"\"Load parish-level data for Table 2.\n\n    Returns:\n        DataFrame with parish turnover data.\n    \"\"\"\n    parish = pd.read_parquet(DATA_CONVERTED / DATA_FILES[\"parish\"])\n    diodata = load_diocese_data()\n\n    # Merge with diocese data\n    df = parish.merge(diodata, on=\"CE_1978\", how=\"left\")\n\n    return df\n\n\ndef load_pt_panel_data() -> pd.DataFrame:\n    \"\"\"Load PT panel data for Table 3.\n\n    Returns:\n        DataFrame with PT affiliation panel.\n    \"\"\"\n    pt_panel = pd.read_parquet(DATA_CONVERTED / DATA_FILES[\"pt_panel\"])\n    diodata = load_diocese_data()\n\n    # Merge with diocese data\n    df = pt_panel.merge(diodata, on=\"CE_1978\", how=\"left\")\n\n    return df\n\n\ndef prepare_analysis_sample(\n    df: pd.DataFrame, exclude_archdioceses: bool = True, election_years: list = None\n) -> pd.DataFrame:\n    \"\"\"Apply sample restrictions and construct derived variables.\n\n    Args:\n        df: Raw loaded DataFrame from load_main_data().\n        exclude_archdioceses: If True, exclude archdioceses (CE_TYPE != \"A\").\n        election_years: List of election years to include.\n\n    Returns:\n        Analysis-ready DataFrame with all filters and transformations applied.\n    \"\"\"\n    df = df.copy()\n\n    # Filter to election years if specified\n    if election_years is not None:\n        df = df[df[\"election\"].isin(election_years)]\n\n    # Filter to PT presidential elections\n    df = df[(df[\"partido\"] == \"PT\") & (df[\"cargo\"] == \"Presidente\")]\n\n    # Exclude archdioceses if requested\n    if exclude_archdioceses:\n        df = df[df[\"CE_TYPE\"] != \"A\"]\n\n    # Construct derived variables\n    df = construct_treatment_variables(df)\n\n    return df\n\n\ndef construct_treatment_variables(df: pd.DataFrame) -> pd.DataFrame:\n    \"\"\"Construct treatment and instrument variables.\n\n    Args:\n        df: DataFrame with raw variables.\n\n    Returns:\n        DataFrame with derived treatment variables.\n    \"\"\"\n    df = df.copy()\n\n    # Retirement_Years (instrument): years since 1978, censored at election\n    df[\"Retirement_Years\"] = df.apply(\n        lambda row: (\n            row[\"I_YEAR\"] - 1978 if row[\"I_YEAR\"] < row[\"election\"] else row[\"election\"] - 1978\n        ),\n        axis=1,\n    )\n\n    # Exposure (treatment): years of JPII bishop exposure, censored at election\n    df[\"Exposure\"] = df.apply(\n        lambda row: (\n            row[\"JPIIAPT_YEAR\"] - 1978\n            if row[\"JPIIAPT_YEAR\"] < row[\"election\"]\n            else row[\"election\"] - 1978\n        ),\n        axis=1,\n    )\n\n    # Alternative treatment for Table C8\n    df[\"Exposure_alt\"] = df.apply(\n        lambda row: (\n            row[\"JPIIELEV_YEAR\"] - 1978\n            if row[\"JPIIELEV_YEAR\"] < row[\"election\"]\n            else row[\"election\"] - 1978\n        ),\n        axis=1,\n    )\n\n    return df\n\n\ndef prepare_parish_sample(df: pd.DataFrame) -> pd.DataFrame:\n    \"\"\"Prepare parish turnover sample for Table 2.\n\n    Args:\n        df: Raw parish data from load_parish_data().\n\n    Returns:\n        Analysis-ready parish sample.\n    \"\"\"\n    df = df.copy()\n\n    # Balanced parish panel: parishes exist in all three years\n    df = df[(df[\"parishes_1965\"] > 0) & (df[\"parishes_1970\"] > 0) & (df[\"parishes_1977\"] > 0)]\n\n    # Exclude archdioceses\n    df = df[df[\"CE_TYPE\"] != \"A\"]\n\n    # Construct treatment variables\n    df[\"I_cons\"] = (df[\"AC_year\"] > df[\"I_YEAR\"]).astype(int)\n    df[\"cons\"] = (df[\"AC_year\"] > df[\"JPIIAPT_YEAR\"]).astype(int)\n\n    return df\n\n\ndef prepare_pt_panel_sample(df: pd.DataFrame) -> pd.DataFrame:\n    \"\"\"Prepare PT panel sample for Table 3.\n\n    Args:\n        df: Raw PT panel data from load_pt_panel_data().\n\n    Returns:\n        Analysis-ready PT panel sample.\n    \"\"\"\n    df = df.copy()\n\n    # Exclude archdioceses and rows with missing diocese linkage\n    df = df[df[\"CE_1978\"].notna() & (df[\"CE_TYPE\"] != \"A\")]\n\n    # Construct treatment variables\n    df[\"I_cons\"] = (df[\"year\"] > df[\"I_YEAR\"]).astype(int)\n    df[\"cons\"] = (df[\"year\"] > df[\"JPIIAPT_YEAR\"]).astype(int)\n\n    # Compute PT office activation threshold\n    # year_start = first year with >= 5 new members\n    df[\"year_start\"] = df.groupby(\"cod.1970\")[\"year\"].transform(\n        lambda years: (\n            years[df.loc[years.index, \"new_members\"] >= 5].min()\n            if (df.loc[years.index, \"new_members\"] >= 5).any()\n            else pd.NA\n        )\n    )\n\n    # had_office: indicator for years after office activation\n    # Create mask where year_start is not null and year > year_start\n    # Use pd.NA-safe comparison\n    has_start = df[\"year_start\"].notna()\n    is_after_start = df[\"year\"] > df[\"year_start\"].where(has_start, df[\"year\"].max() + 1)\n    df[\"had_office\"] = (has_start & is_after_start).astype(int)\n\n    return df\n\n\ndef get_sample_stats(sample: pd.DataFrame) -> dict:\n    \"\"\"Compute summary statistics for the analysis sample.\n\n    Args:\n        sample: Prepared analysis sample from prepare_analysis_sample().\n\n    Returns:\n        Dictionary with keys like n_obs, n_units, year_range, etc.\n    \"\"\"\n    stats = {\n        \"n_obs\": len(sample),\n        \"n_dioceses\": sample[\"CE_1978\"].nunique() if \"CE_1978\" in sample.columns else None,\n        \"n_municipalities\": (\n            sample[\"cod.2010\"].nunique() if \"cod.2010\" in sample.columns else None\n        ),\n        \"n_states\": sample[\"UF\"].nunique() if \"UF\" in sample.columns else None,\n        \"election_years\": (\n            sorted(sample[\"election\"].unique().tolist()) if \"election\" in sample.columns else None\n        ),\n        \"mean_vote_share\": (\n            sample[\"vote.sh\"].mean() if \"vote.sh\" in sample.columns else None\n        ),\n        \"mean_exposure\": sample[\"Exposure\"].mean() if \"Exposure\" in sample.columns else None,\n        \"mean_retirement_years\": (\n            sample[\"Retirement_Years\"].mean() if \"Retirement_Years\" in sample.columns else None\n        ),\n    }\n\n    return stats\n")
Path("src/descriptives.py").write_text("\"\"\"Descriptive analyses: Figures 1, 3, Table A1.\"\"\"\n\nimport matplotlib.pyplot as plt\nimport pandas as pd\n\nfrom .config import ELECTION_YEARS\nfrom .data import load_main_data, prepare_analysis_sample\nfrom .helpers import save_figure, save_table_latex\n\n\ndef figure1(sample: pd.DataFrame = None) -> None:\n    \"\"\"Figure 1: Maps of PT vote share by municipality (1989-2002).\n\n    Note: This requires geospatial data (shapefiles). For now, we create\n    a placeholder showing vote share distributions.\n    \"\"\"\n    if sample is None:\n        df = load_main_data()\n        sample = prepare_analysis_sample(df, exclude_archdioceses=True)\n\n    fig, axes = plt.subplots(2, 2, figsize=(12, 10))\n    axes = axes.flatten()\n\n    for i, year in enumerate(ELECTION_YEARS):\n        df_year = sample[sample[\"election\"] == year]\n\n        # Create histogram of vote shares\n        axes[i].hist(df_year[\"vote.sh\"], bins=30, edgecolor=\"black\", alpha=0.7)\n        axes[i].set_xlabel(\"PT Vote Share (%)\")\n        axes[i].set_ylabel(\"Frequency\")\n        axes[i].set_title(f\"{year} Presidential Election\")\n        axes[i].axvline(\n            df_year[\"vote.sh\"].mean(), color=\"red\", linestyle=\"--\", linewidth=2, label=\"Mean\"\n        )\n        axes[i].legend()\n\n    plt.suptitle(\"Figure 1: PT Vote Share Distribution (1989-2002)\", fontsize=14, y=1.00)\n    plt.tight_layout()\n    save_figure(fig, \"figure1.pdf\")\n    plt.close()\n\n    print(\"Note: Figure 1 requires geospatial shapefiles for mapping. Created distribution plots.\")\n\n\ndef figure3(sample: pd.DataFrame = None) -> None:\n    \"\"\"Figure 3: Maps of mandated exposure to progressive bishops (1989-2002).\n\n    Note: This also requires geospatial data. For now, we create distribution plots.\n    \"\"\"\n    if sample is None:\n        df = load_main_data()\n        sample = prepare_analysis_sample(df, exclude_archdioceses=True)\n\n    fig, axes = plt.subplots(2, 2, figsize=(12, 10))\n    axes = axes.flatten()\n\n    for i, year in enumerate(ELECTION_YEARS):\n        df_year = sample[sample[\"election\"] == year]\n\n        # Create histogram of Retirement_Years\n        axes[i].hist(df_year[\"Retirement_Years\"], bins=20, edgecolor=\"black\", alpha=0.7)\n        axes[i].set_xlabel(\"Mandated Retirement Years\")\n        axes[i].set_ylabel(\"Frequency\")\n        axes[i].set_title(f\"{year} Election\")\n        axes[i].axvline(\n            df_year[\"Retirement_Years\"].mean(),\n            color=\"red\",\n            linestyle=\"--\",\n            linewidth=2,\n            label=\"Mean\",\n        )\n        axes[i].legend()\n\n    plt.suptitle(\n        \"Figure 3: Mandated Exposure Distribution (1989-2002)\", fontsize=14, y=1.00\n    )\n    plt.tight_layout()\n    save_figure(fig, \"figure3.pdf\")\n    plt.close()\n\n    print(\"Note: Figure 3 requires geospatial shapefiles for mapping. Created distribution plots.\")\n\n\ndef tableA1(sample: pd.DataFrame = None) -> pd.DataFrame:\n    \"\"\"Table A1: Summary statistics.\n\n    Descriptive statistics for key variables.\n    \"\"\"\n    if sample is None:\n        df = load_main_data()\n        sample = prepare_analysis_sample(df, exclude_archdioceses=True)\n\n    # Key variables to summarize\n    vars_to_summarize = [\n        \"vote.sh\",\n        \"Exposure\",\n        \"Retirement_Years\",\n        \"pop.tot.1970\",\n        \"pop.urb.1970\",\n        \"share_catolico_1978\",\n        \"share_evangelico_1978\",\n        \"ELEITORADO_1976\",\n        \"MDB1976_SH\",\n    ]\n\n    # Collapse to municipality level (one row per municipality)\n    muni_level = sample.groupby(\"cod.2010\").first().reset_index()\n\n    summary_stats = []\n\n    for var in vars_to_summarize:\n        if var in muni_level.columns:\n            series = muni_level[var].dropna()\n\n            summary_stats.append(\n                {\n                    \"Variable\": var,\n                    \"N\": len(series),\n                    \"Mean\": f\"{series.mean():.2f}\",\n                    \"SD\": f\"{series.std():.2f}\",\n                    \"Min\": f\"{series.min():.2f}\",\n                    \"Max\": f\"{series.max():.2f}\",\n                }\n            )\n\n    table = pd.DataFrame(summary_stats)\n\n    save_table_latex(\n        table, \"tableA1.tex\", caption=\"Summary Statistics\", label=\"tab:summary\"\n    )\n\n    return table\n\n\ndef run(sample: pd.DataFrame = None):\n    \"\"\"Run all descriptive analyses.\"\"\"\n    print(\"Running Figure 1 (PT Vote Share Maps)...\")\n    figure1(sample)\n\n    print(\"Running Figure 3 (Mandated Exposure Maps)...\")\n    figure3(sample)\n\n    print(\"Running Table A1 (Summary Statistics)...\")\n    tableA1(sample)\n\n    print(\"Descriptive analyses complete.\")\n")
Path("src/helpers.py").write_text("\"\"\"Helper functions for regression output formatting.\"\"\"\n\nfrom typing import Any, Optional\n\nimport numpy as np\nimport pandas as pd\n\n\ndef format_coefficient(coef: float, se: float, stars: bool = True) -> tuple:\n    \"\"\"Format coefficient and standard error for table output.\n\n    Args:\n        coef: Coefficient estimate\n        se: Standard error\n        stars: If True, add significance stars\n\n    Returns:\n        Tuple of (formatted_coef, formatted_se)\n    \"\"\"\n    # Determine significance stars\n    star_str = \"\"\n    if stars:\n        t_stat = abs(coef / se) if se > 0 else 0\n        if t_stat > 2.576:  # 99% confidence\n            star_str = \"***\"\n        elif t_stat > 1.96:  # 95% confidence\n            star_str = \"**\"\n        elif t_stat > 1.645:  # 90% confidence\n            star_str = \"*\"\n\n    coef_str = f\"{coef:.3f}{star_str}\"\n    se_str = f\"({se:.3f})\"\n\n    return coef_str, se_str\n\n\ndef extract_pyfixest_results(fit, var_name: str = None) -> dict[str, Any]:\n    \"\"\"Extract regression results from pyfixest fit object.\n\n    Args:\n        fit: pyfixest fit object\n        var_name: Variable name to extract (if None, uses first covariate)\n\n    Returns:\n        Dictionary with coefficient, se, pval, nobs, nclusters, etc.\n    \"\"\"\n    if var_name is None:\n        var_name = fit.coef().index[0]\n\n    results = {\n        \"coef\": fit.coef().loc[var_name],\n        \"se\": fit.se().loc[var_name],\n        \"pval\": fit.pvalue().loc[var_name],\n        \"tstat\": fit.tstat().loc[var_name],\n        \"nobs\": fit.nobs,\n        \"r2\": fit.r2,\n    }\n\n    # Extract clustering info if available\n    if hasattr(fit, \"_vcov_type_detail\") and fit._vcov_type_detail:\n        results[\"cluster_var\"] = fit._vcov_type_detail.get(\"CRV1\", None)\n\n    return results\n\n\ndef build_regression_table(\n    results_list: list[dict[str, Any]],\n    outcome_mean: Optional[list[float]] = None,\n    panel_labels: Optional[list[str]] = None,\n) -> pd.DataFrame:\n    \"\"\"Build a regression table from list of results.\n\n    Args:\n        results_list: List of result dictionaries from extract_pyfixest_results()\n        outcome_mean: List of outcome means for each column\n        panel_labels: Optional labels for panels (e.g., [\"Panel A: 2SLS\", \"Panel B: ITT\"])\n\n    Returns:\n        DataFrame formatted as regression table\n    \"\"\"\n    n_cols = len(results_list)\n    table_data = []\n\n    # Add panel label if provided\n    if panel_labels and len(panel_labels) > 0:\n        for label in panel_labels:\n            table_data.append([label] + [\"\"] * (n_cols - 1))\n\n    # Coefficient row\n    coef_row = []\n    se_row = []\n    for res in results_list:\n        coef_str, se_str = format_coefficient(res[\"coef\"], res[\"se\"])\n        coef_row.append(coef_str)\n        se_row.append(se_str)\n\n    table_data.append(coef_row)\n    table_data.append(se_row)\n\n    # Outcome mean\n    if outcome_mean:\n        table_data.append([\"Outcome mean\"] + [f\"{m:.2f}\" for m in outcome_mean])\n\n    # N observations\n    table_data.append([\"N obs\"] + [str(res[\"nobs\"]) for res in results_list])\n\n    # N clusters (if available)\n    if \"nclusters\" in results_list[0]:\n        table_data.append([\"N clusters\"] + [str(res[\"nclusters\"]) for res in results_list])\n\n    # R-squared\n    table_data.append([\"R²\"] + [f\"{res['r2']:.3f}\" for res in results_list])\n\n    return pd.DataFrame(table_data)\n\n\ndef save_table_latex(\n    table: pd.DataFrame, filename: str, caption: str = \"\", label: str = \"\"\n) -> None:\n    \"\"\"Save table as LaTeX file.\n\n    Args:\n        table: DataFrame to save\n        filename: Output filename (should end in .tex)\n        caption: Table caption\n        label: LaTeX label for referencing\n    \"\"\"\n    from .config import TABLE_DIR\n\n    TABLE_DIR.mkdir(parents=True, exist_ok=True)\n    output_path = TABLE_DIR / filename\n\n    # Convert to LaTeX\n    latex_str = table.to_latex(index=False, escape=False)\n\n    # Add caption and label if provided\n    if caption or label:\n        lines = latex_str.split(\"\\n\")\n        # Insert after \\begin{tabular}\n        for i, line in enumerate(lines):\n            if \"\\\\begin{tabular}\" in line:\n                if caption:\n                    lines.insert(i, f\"\\\\caption{{{caption}}}\")\n                if label:\n                    lines.insert(i + 1, f\"\\\\label{{{label}}}\")\n                break\n        latex_str = \"\\n\".join(lines)\n\n    output_path.write_text(latex_str)\n    print(f\"Saved: {output_path}\")\n\n\ndef save_figure(fig, filename: str, dpi: int = 300) -> None:\n    \"\"\"Save matplotlib figure.\n\n    Args:\n        fig: Matplotlib figure object\n        filename: Output filename (should end in .pdf or .png)\n        dpi: Resolution for raster formats\n    \"\"\"\n    from .config import FIGURE_DIR\n\n    FIGURE_DIR.mkdir(parents=True, exist_ok=True)\n    output_path = FIGURE_DIR / filename\n\n    fig.savefig(output_path, dpi=dpi, bbox_inches=\"tight\")\n    print(f\"Saved: {output_path}\")\n\n\ndef compute_first_stage_fstat(fit) -> float:\n    \"\"\"Compute first-stage F-statistic from IV regression.\n\n    Args:\n        fit: pyfixest IV fit object\n\n    Returns:\n        First-stage F-statistic\n    \"\"\"\n    # pyfixest stores first-stage F-stat in _iv_diagnostics\n    if hasattr(fit, \"_iv_diagnostics\") and fit._iv_diagnostics:\n        return fit._iv_diagnostics.get(\"first_stage_fstat\", np.nan)\n\n    # Fallback: compute manually if not available\n    return np.nan\n\n\ndef stars_from_pval(pval: float) -> str:\n    \"\"\"Convert p-value to significance stars.\n\n    Args:\n        pval: P-value\n\n    Returns:\n        String with stars: \"***\", \"**\", \"*\", or \"\"\n    \"\"\"\n    if pval < 0.01:\n        return \"***\"\n    elif pval < 0.05:\n        return \"**\"\n    elif pval < 0.10:\n        return \"*\"\n    else:\n        return \"\"\n\n\ndef aggregate_to_diocese(\n    df: pd.DataFrame, outcome_vars: list[str], diocese_var: str = \"CE_code\"\n) -> pd.DataFrame:\n    \"\"\"Aggregate municipality-level data to diocese level.\n\n    Args:\n        df: Municipality-level DataFrame\n        outcome_vars: List of outcome variables to aggregate\n        diocese_var: Diocese identifier variable\n\n    Returns:\n        Diocese-level DataFrame\n    \"\"\"\n    # Group by diocese and other grouping vars\n    group_vars = [diocese_var, \"cargo\", \"election\", \"partido\"]\n    group_vars = [v for v in group_vars if v in df.columns]\n\n    # Sum vote counts\n    agg_dict = {v: \"sum\" for v in outcome_vars if v in df.columns}\n\n    # Aggregate\n    df_dio = df.groupby(group_vars, as_index=False).agg(agg_dict)\n\n    return df_dio\n")
Path("src/tests_of_design.py").write_text("\"\"\"Tests of design: Figure 2, Tables B1, B2.\"\"\"\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport pyfixest as pf\n\nfrom .data import load_diocese_data, load_main_data\nfrom .helpers import extract_pyfixest_results, save_figure, save_table_latex\n\n\ndef figure2(sample: pd.DataFrame = None) -> None:\n    \"\"\"Figure 2: First-stage analysis.\n\n    Panel A: Scatterplot of Mandated Retirement vs JPII Appointment\n    Panel B: First-stage coefficient over time\n    Panel C: First-stage F-statistic over time\n    \"\"\"\n    diodata = load_diocese_data()\n\n    # Years 1979-2020\n    years = range(1979, 2021)\n    coefficients = []\n    fstats = []\n\n    for year in years:\n        # Construct treatment variables for this year\n        I_years = np.where(\n            diodata[\"I_YEAR\"] < year, diodata[\"I_YEAR\"] - 1978, year - 1978\n        )\n        Exposure = np.where(\n            diodata[\"JPIIAPT_YEAR\"] < year, diodata[\"JPIIAPT_YEAR\"] - 1978, year - 1978\n        )\n\n        # First-stage regression\n        df_year = pd.DataFrame({\"Exposure\": Exposure, \"I_years\": I_years})\n        fit = pf.feols(\"Exposure ~ I_years\", data=df_year, vcov=\"iid\")\n\n        coefficients.append(fit.coef().loc[\"I_years\"])\n        fstats.append(fit.tstat().loc[\"I_years\"] ** 2)  # F-stat = t-stat^2 for single regressor\n\n    # Create figure\n    fig, axes = plt.subplots(1, 3, figsize=(15, 4))\n\n    # Panel A: Scatterplot (for 2002)\n    I_2002 = np.where(diodata[\"I_YEAR\"] < 2002, diodata[\"I_YEAR\"] - 1978, 2002 - 1978)\n    Exp_2002 = np.where(diodata[\"JPIIAPT_YEAR\"] < 2002, diodata[\"JPIIAPT_YEAR\"] - 1978, 2002 - 1978)\n    axes[0].scatter(I_2002, Exp_2002, alpha=0.6)\n    axes[0].plot([0, 24], [0, 24], \"r--\", alpha=0.5)\n    axes[0].set_xlabel(\"Mandated Retirement Years (2002)\")\n    axes[0].set_ylabel(\"JPII Exposure Years (2002)\")\n    axes[0].set_title(\"Panel A: First-Stage Relationship\")\n\n    # Panel B: Coefficient over time\n    axes[1].plot(years, coefficients, linewidth=2)\n    axes[1].axhline(y=1.0, color=\"r\", linestyle=\"--\", alpha=0.5)\n    axes[1].set_xlabel(\"Year\")\n    axes[1].set_ylabel(\"First-Stage Coefficient\")\n    axes[1].set_title(\"Panel B: Coefficient Over Time\")\n\n    # Panel C: F-statistic over time\n    axes[2].plot(years, fstats, linewidth=2)\n    axes[2].axhline(y=10, color=\"r\", linestyle=\"--\", alpha=0.5, label=\"Weak IV threshold\")\n    axes[2].set_xlabel(\"Year\")\n    axes[2].set_ylabel(\"First-Stage F-Statistic\")\n    axes[2].set_title(\"Panel C: F-Statistic Over Time\")\n    axes[2].legend()\n\n    plt.tight_layout()\n    save_figure(fig, \"figure2.pdf\")\n    plt.close()\n\n\ndef tableB1(sample: pd.DataFrame = None) -> pd.DataFrame:\n    \"\"\"Table B1: Balance tests (diocese and municipality characteristics).\"\"\"\n    diodata = load_diocese_data()\n    df = load_main_data()\n\n    # Collapse to municipality level (one obs per municipality)\n    munidata = df.groupby(\"cod.2010\").first().reset_index()\n\n    # Panel A: Diocese-level variables\n    dio_vars = [\n        \"IR\", \"NIR\", \"INR\", \"DIAC\", \"REL\", \"CASASREL_MASC\", \"CASASREL_FEM\",\n        \"NR_PAR\", \"NR_MUNI\", \"recursos_nro\", \"recursos_total_activities\",\n    ]\n\n    # Panel B: Municipality-level variables\n    muni_vars = [\n        \"pop.tot.1970\", \"pop.urb.1970\", \"pop.rur.1970\",\n        \"share_catolico_1970\", \"share_evangelico_1970\",\n        \"share_catolico_1978\", \"share_evangelico_1978\", \"growth_evangelica\",\n    ]\n\n    # Panel C: Pre-treatment electoral variables\n    electoral_vars = [\n        \"ELEITORADO_1972\", \"MDB1972_SH\", \"ARENA1972_SH\",\n        \"ELEITORADO_1976\", \"MDB1976_SH\", \"ARENA1976_SH\",\n    ]\n\n    results = []\n\n    # Run balance tests: I_YEAR ~ X\n    # Diocese level (no clustering)\n    for var in dio_vars:\n        if var in diodata.columns:\n            fit = pf.feols(f\"I_YEAR ~ {var}\", data=diodata, vcov=\"iid\")\n            results.append(extract_balance_result(var, fit))\n\n    # Municipality level (with state FE and diocese clustering)\n    for var in muni_vars + electoral_vars:\n        if var in munidata.columns:\n            fit = pf.feols(\n                f\"I_YEAR ~ {var} | UF\", data=munidata, vcov={\"CRV1\": \"CE_1978\"}\n            )\n            results.append(extract_balance_result(var, fit))\n\n    # Build table\n    table = pd.DataFrame(results)\n    save_table_latex(\n        table, \"tableB1.tex\", caption=\"Balance Tests\", label=\"tab:balance\"\n    )\n\n    return table\n\n\ndef extract_balance_result(var_name: str, fit) -> dict:\n    \"\"\"Extract balance test result.\"\"\"\n    try:\n        res = extract_pyfixest_results(fit, var_name)\n        return {\n            \"Variable\": var_name,\n            \"Coefficient\": f\"{res['coef']:.3f}\",\n            \"SE\": f\"({res['se']:.3f})\",\n            \"P-value\": f\"{res['pval']:.3f}\",\n        }\n    except Exception:\n        return {\n            \"Variable\": var_name,\n            \"Coefficient\": \"NA\",\n            \"SE\": \"NA\",\n            \"P-value\": \"NA\",\n        }\n\n\ndef tableB2(sample: pd.DataFrame = None) -> pd.DataFrame:\n    \"\"\"Table B2: Placebo test - MDB 1976 vote share (pre-treatment).\"\"\"\n    df = load_main_data()\n\n    # Collapse to municipality level\n    munidata = df.groupby(\"cod.2010\").first().reset_index()\n\n    # Exclude archdioceses\n    munidata = munidata[munidata[\"CE_TYPE\"] != \"A\"]\n\n    # Drop missing MDB1976_SH\n    munidata = munidata[munidata[\"MDB1976_SH\"].notna()]\n\n    # Compute Retirement_Years for 1976 (pre-treatment placebo)\n    munidata[\"Retirement_1976\"] = np.where(\n        munidata[\"I_YEAR\"] < 1976, munidata[\"I_YEAR\"] - 1978, 1976 - 1978\n    )\n\n    # Run placebo regression: MDB1976_SH ~ Retirement_1976 | UF | cluster(CE_1978)\n    fit = pf.feols(\n        \"MDB1976_SH ~ Retirement_1976 | UF\",\n        data=munidata,\n        vcov={\"CRV1\": \"CE_1978\"},\n    )\n\n    res = extract_pyfixest_results(fit, \"Retirement_1976\")\n\n    # Build table\n    rows = [\n        [\"Outcome: MDB 1976 Vote Share\"],\n        [f\"Retirement Years: {res['coef']:.3f}{get_stars(res['pval'])}\"],\n        [f\"({res['se']:.3f})\"],\n        [\"\"],\n        [f\"N obs: {res['nobs']}\"],\n        [\"State FE: Yes\"],\n        [\"SE: Clustered (Diocese)\"],\n    ]\n\n    table = pd.DataFrame(rows)\n    save_table_latex(\n        table, \"tableB2.tex\", caption=\"Placebo Test (Pre-Treatment)\", label=\"tab:placebo\"\n    )\n\n    return table\n\n\ndef figureB2(sample: pd.DataFrame = None) -> None:\n    \"\"\"Figure B2: First stage by exit type (retirement/death/etc.).\"\"\"\n    diodata = load_diocese_data()\n\n    exit_types = diodata[\"out_method_0\"].unique()\n    exit_types = [e for e in exit_types if pd.notna(e)]\n\n    fig, ax = plt.subplots(figsize=(8, 6))\n\n    for exit_type in exit_types:\n        dio_subset = diodata[diodata[\"out_method_0\"] == exit_type]\n\n        years = range(1979, 2021)\n        coefficients = []\n\n        for year in years:\n            I_years = np.where(\n                dio_subset[\"I_YEAR\"] < year, dio_subset[\"I_YEAR\"] - 1978, year - 1978\n            )\n            Exposure = np.where(\n                dio_subset[\"JPIIAPT_YEAR\"] < year,\n                dio_subset[\"JPIIAPT_YEAR\"] - 1978,\n                year - 1978,\n            )\n\n            df_year = pd.DataFrame({\"Exposure\": Exposure, \"I_years\": I_years})\n            fit = pf.feols(\"Exposure ~ I_years\", data=df_year, vcov=\"iid\")\n            coefficients.append(fit.coef().loc[\"I_years\"])\n\n        ax.plot(years, coefficients, label=exit_type, linewidth=2)\n\n    ax.axhline(y=1.0, color=\"gray\", linestyle=\"--\", alpha=0.5)\n    ax.set_xlabel(\"Year\")\n    ax.set_ylabel(\"First-Stage Coefficient\")\n    ax.set_title(\"First-Stage by Bishop Exit Type\")\n    ax.legend()\n\n    save_figure(fig, \"figureB2.pdf\")\n    plt.close()\n\n\ndef figureB3(sample: pd.DataFrame = None) -> None:\n    \"\"\"Figure B3: Age distribution of bishops (October 1978).\"\"\"\n    diodata = load_diocese_data()\n\n    fig, ax = plt.subplots(figsize=(8, 6))\n\n    # Separate archdioceses vs dioceses\n    arch = diodata[diodata[\"CE_TYPE\"] == \"A\"][\"AGEOCT78\"]\n    dio = diodata[diodata[\"CE_TYPE\"] != \"A\"][\"AGEOCT78\"]\n\n    ax.hist(dio, bins=15, alpha=0.6, label=\"Dioceses\", edgecolor=\"black\")\n    ax.hist(arch, bins=15, alpha=0.6, label=\"Archdioceses\", edgecolor=\"black\")\n    ax.axvline(x=75, color=\"red\", linestyle=\"--\", linewidth=2, label=\"Mandatory Retirement Age\")\n    ax.set_xlabel(\"Bishop Age (October 1978)\")\n    ax.set_ylabel(\"Frequency\")\n    ax.set_title(\"Age Distribution of Brazilian Bishops\")\n    ax.legend()\n\n    save_figure(fig, \"figureB3.pdf\")\n    plt.close()\n\n\ndef get_stars(pval: float) -> str:\n    \"\"\"Get significance stars from p-value.\"\"\"\n    if pval < 0.01:\n        return \"***\"\n    elif pval < 0.05:\n        return \"**\"\n    elif pval < 0.10:\n        return \"*\"\n    return \"\"\n\n\ndef run(sample: pd.DataFrame = None):\n    \"\"\"Run all tests of design.\"\"\"\n    print(\"Running Figure 2 (First-Stage Analysis)...\")\n    figure2(sample)\n\n    print(\"Running Table B1 (Balance Tests)...\")\n    tableB1(sample)\n\n    print(\"Running Table B2 (Placebo Test)...\")\n    tableB2(sample)\n\n    print(\"Running Figure B2 (First-Stage by Exit Type)...\")\n    figureB2(sample)\n\n    print(\"Running Figure B3 (Bishop Age Distribution)...\")\n    figureB3(sample)\n\n    print(\"Tests of design complete.\")\n")

# Ensure src is importable
sys.path.insert(0, ".")

# Create output directories
Path("output/tables").mkdir(parents=True, exist_ok=True)
Path("output/figures").mkdir(parents=True, exist_ok=True)

## Data Preparation


### Load Primary Datasets

Load municipality characteristics, diocese characteristics, and electoral
outcome data from converted Parquet files.

In [ ]:
def load_main_data() -> pd.DataFrame:
    """Load the main analysis dataset from converted data.

    Returns:
        DataFrame with all variables needed for analysis.
    """
    # Load primary datasets
    munidata = pd.read_parquet(DATA_CONVERTED / DATA_FILES["munidata"])
    electoral = pd.read_parquet(DATA_CONVERTED / DATA_FILES["electoral"])

    # Merge municipality data with electoral data
    df = munidata.merge(electoral, on="cod.2010", how="left")

    # Return merged dataset
    return df

def load_diocese_data() -> pd.DataFrame:
    """Load diocese-level data.

    Returns:
        DataFrame with diocese characteristics.
    """
    return pd.read_parquet(DATA_CONVERTED / DATA_FILES["diodata"])

In [ ]:
from src.data import load_main_data, load_diocese_data
munidata = pd.read_parquet(DATA_CONVERTED / "munidata.parquet")
diodata = load_diocese_data()
electoral = pd.read_parquet(DATA_CONVERTED / "IPEA_electoral_data.parquet")
print(f"munidata: {munidata.shape}")
print(f"diodata: {diodata.shape}")
print(f"electoral: {electoral.shape}")
display(munidata.head())


### Build Municipality-Election Panel

Merge municipality data with electoral outcomes on cod.2010, creating the
main analysis panel for electoral analysis (Tables 1, C1-C8).

In [ ]:
def load_main_data() -> pd.DataFrame:
    """Load the main analysis dataset from converted data.

    Returns:
        DataFrame with all variables needed for analysis.
    """
    # Load primary datasets
    munidata = pd.read_parquet(DATA_CONVERTED / DATA_FILES["munidata"])
    electoral = pd.read_parquet(DATA_CONVERTED / DATA_FILES["electoral"])

    # Merge municipality data with electoral data
    df = munidata.merge(electoral, on="cod.2010", how="left")

    # Return merged dataset
    return df

In [ ]:
df = load_main_data()
print(f"Municipality-election panel: {df.shape}")
print(f"Elections: {sorted(df['election'].dropna().unique())}")
print(f"Parties: {df['partido'].nunique()}")
display(df[['cod.2010', 'election', 'partido', 'cargo', 'votos', 'vote.sh']].head(10))


### Construct Electoral Treatment Variables

Create Retirement_Years (instrument) and Exposure (treatment) using censored
timing logic. If bishop retirement/appointment occurred before the election,
treatment equals years since 1978; otherwise censored at election year.

In [ ]:
def construct_treatment_variables(df: pd.DataFrame) -> pd.DataFrame:
    """Construct treatment and instrument variables.

    Args:
        df: DataFrame with raw variables.

    Returns:
        DataFrame with derived treatment variables.
    """
    df = df.copy()

    # Retirement_Years (instrument): years since 1978, censored at election
    df["Retirement_Years"] = df.apply(
        lambda row: (
            row["I_YEAR"] - 1978 if row["I_YEAR"] < row["election"] else row["election"] - 1978
        ),
        axis=1,
    )

    # Exposure (treatment): years of JPII bishop exposure, censored at election
    df["Exposure"] = df.apply(
        lambda row: (
            row["JPIIAPT_YEAR"] - 1978
            if row["JPIIAPT_YEAR"] < row["election"]
            else row["election"] - 1978
        ),
        axis=1,
    )

    # Alternative treatment for Table C8
    df["Exposure_alt"] = df.apply(
        lambda row: (
            row["JPIIELEV_YEAR"] - 1978
            if row["JPIIELEV_YEAR"] < row["election"]
            else row["election"] - 1978
        ),
        axis=1,
    )

    return df

In [ ]:
from src.data import prepare_analysis_sample
sample = prepare_analysis_sample(load_main_data(), election_years=[1989, 1994, 1998, 2002])
print(f"Analysis sample: {sample.shape}")
print(f"Mean Retirement_Years: {sample['Retirement_Years'].mean():.2f}")
print(f"Mean Exposure: {sample['Exposure'].mean():.2f}")
display(sample[['election', 'I_YEAR', 'JPIIAPT_YEAR', 'Retirement_Years', 'Exposure']].head(10))


### Apply Electoral Sample Restrictions

Filter to PT presidential elections (1989, 1994, 1998, 2002) and exclude
archdioceses (CE_TYPE != 'A'). This produces the main analysis sample for
Table 1.

In [ ]:
def prepare_analysis_sample(
    df: pd.DataFrame, exclude_archdioceses: bool = True, election_years: list = None
) -> pd.DataFrame:
    """Apply sample restrictions and construct derived variables.

    Args:
        df: Raw loaded DataFrame from load_main_data().
        exclude_archdioceses: If True, exclude archdioceses (CE_TYPE != "A").
        election_years: List of election years to include.

    Returns:
        Analysis-ready DataFrame with all filters and transformations applied.
    """
    df = df.copy()

    # Filter to election years if specified
    if election_years is not None:
        df = df[df["election"].isin(election_years)]

    # Filter to PT presidential elections
    df = df[(df["partido"] == "PT") & (df["cargo"] == "Presidente")]

    # Exclude archdioceses if requested
    if exclude_archdioceses:
        df = df[df["CE_TYPE"] != "A"]

    # Construct derived variables
    df = construct_treatment_variables(df)

    return df

In [ ]:
sample = prepare_analysis_sample(
    load_main_data(),
    exclude_archdioceses=True,
    election_years=[1989, 1994, 1998, 2002]
)
print(f"Analysis sample: {sample.shape}")
print(f"Dioceses: {sample['CE_1978'].nunique()}")
print(f"Municipalities: {sample['cod.2010'].nunique()}")
print(f"Mean PT vote share: {sample['vote.sh'].mean():.2f}%")
display(sample.groupby('election').size())


### Build Diocese-Level Panel

Aggregate municipality votes to diocese level for Table C1 robustness check.
Groups by diocese, election, office, and party; sums votes and computes
diocese-level vote shares.

In [ ]:
def load_main_data() -> pd.DataFrame:
    """Load the main analysis dataset from converted data.

    Returns:
        DataFrame with all variables needed for analysis.
    """
    # Load primary datasets
    munidata = pd.read_parquet(DATA_CONVERTED / DATA_FILES["munidata"])
    electoral = pd.read_parquet(DATA_CONVERTED / DATA_FILES["electoral"])

    # Merge municipality data with electoral data
    df = munidata.merge(electoral, on="cod.2010", how="left")

    # Return merged dataset
    return df

def load_diocese_data() -> pd.DataFrame:
    """Load diocese-level data.

    Returns:
        DataFrame with diocese characteristics.
    """
    return pd.read_parquet(DATA_CONVERTED / DATA_FILES["diodata"])

In [ ]:
munidata = load_main_data()
diodata = load_diocese_data()
# Aggregate to diocese level
dio_panel = munidata.groupby(['CE_code', 'cargo', 'election', 'partido']).agg({
    'votos': 'sum',
    'votos.sum': 'sum'
}).reset_index()
dio_panel['vote.sh'] = 100 * dio_panel['votos'] / dio_panel['votos.sum']
dio_panel = dio_panel.merge(diodata, on='CE_code', how='left')
print(f"Diocese panel: {dio_panel.shape}")
print(f"Dioceses: {dio_panel['CE_code'].nunique()}")
display(dio_panel.head())


### Load Parish Turnover Data

Load parish-level data with yearbook observations. Used for Table 2 mechanism
analysis showing progressive bishops increased priest turnover.

In [ ]:
def load_parish_data() -> pd.DataFrame:
    """Load parish-level data for Table 2.

    Returns:
        DataFrame with parish turnover data.
    """
    parish = pd.read_parquet(DATA_CONVERTED / DATA_FILES["parish"])
    diodata = load_diocese_data()

    # Merge with diocese data
    df = parish.merge(diodata, on="CE_1978", how="left")

    return df

In [ ]:
from src.data import load_parish_data
parish = load_parish_data()
print(f"Parish data: {parish.shape}")
print(f"Yearbooks: {sorted(parish['AC_year'].unique())}")
print(f"Dioceses: {parish['CE_1978'].nunique()}")
display(parish.head())


### Prepare Parish Sample

Apply balanced parish panel filter (parishes exist in 1965, 1970, 1977),
exclude archdioceses, and construct binary treatment variables (I_cons,
cons) for Table 2.

In [ ]:
def prepare_parish_sample(df: pd.DataFrame) -> pd.DataFrame:
    """Prepare parish turnover sample for Table 2.

    Args:
        df: Raw parish data from load_parish_data().

    Returns:
        Analysis-ready parish sample.
    """
    df = df.copy()

    # Balanced parish panel: parishes exist in all three years
    df = df[(df["parishes_1965"] > 0) & (df["parishes_1970"] > 0) & (df["parishes_1977"] > 0)]

    # Exclude archdioceses
    df = df[df["CE_TYPE"] != "A"]

    # Construct treatment variables
    df["I_cons"] = (df["AC_year"] > df["I_YEAR"]).astype(int)
    df["cons"] = (df["AC_year"] > df["JPIIAPT_YEAR"]).astype(int)

    return df

In [ ]:
from src.data import prepare_parish_sample, load_parish_data
parish_sample = prepare_parish_sample(load_parish_data())
print(f"Parish sample: {parish_sample.shape}")
print(f"Mean I_cons: {parish_sample['I_cons'].mean():.2%}")
print(f"Mean cons: {parish_sample['cons'].mean():.2%}")
print(f"Turnover observations: {parish_sample['turnover_priest_01'].notna().sum()}")
display(parish_sample[['AC_year', 'I_YEAR', 'JPIIAPT_YEAR', 'I_cons', 'cons', 'turnover_priest_01']].head(10))


### Load PT Local Presence Panel

Load PT affiliation panel tracking new member registrations by municipality
and year. Used for Table 3 mechanism analysis.

In [ ]:
def load_pt_panel_data() -> pd.DataFrame:
    """Load PT panel data for Table 3.

    Returns:
        DataFrame with PT affiliation panel.
    """
    pt_panel = pd.read_parquet(DATA_CONVERTED / DATA_FILES["pt_panel"])
    diodata = load_diocese_data()

    # Merge with diocese data
    df = pt_panel.merge(diodata, on="CE_1978", how="left")

    return df

In [ ]:
from src.data import load_pt_panel_data
pt_panel = load_pt_panel_data()
print(f"PT panel: {pt_panel.shape}")
print(f"Years: {int(pt_panel['year'].min())}-{int(pt_panel['year'].max())}")
print(f"Municipalities: {pt_panel['cod.1970'].nunique()}")
display(pt_panel.head())


### Prepare PT Panel Sample

Construct PT office activation indicator (threshold: 5+ new members) and
binary treatment variables. Excludes archdioceses and municipalities without
diocese linkage.

In [ ]:
def prepare_pt_panel_sample(df: pd.DataFrame) -> pd.DataFrame:
    """Prepare PT panel sample for Table 3.

    Args:
        df: Raw PT panel data from load_pt_panel_data().

    Returns:
        Analysis-ready PT panel sample.
    """
    df = df.copy()

    # Exclude archdioceses and rows with missing diocese linkage
    df = df[df["CE_1978"].notna() & (df["CE_TYPE"] != "A")]

    # Construct treatment variables
    df["I_cons"] = (df["year"] > df["I_YEAR"]).astype(int)
    df["cons"] = (df["year"] > df["JPIIAPT_YEAR"]).astype(int)

    # Compute PT office activation threshold
    # year_start = first year with >= 5 new members
    df["year_start"] = df.groupby("cod.1970")["year"].transform(
        lambda years: (
            years[df.loc[years.index, "new_members"] >= 5].min()
            if (df.loc[years.index, "new_members"] >= 5).any()
            else pd.NA
        )
    )

    # had_office: indicator for years after office activation
    # Create mask where year_start is not null and year > year_start
    # Use pd.NA-safe comparison
    has_start = df["year_start"].notna()
    is_after_start = df["year"] > df["year_start"].where(has_start, df["year"].max() + 1)
    df["had_office"] = (has_start & is_after_start).astype(int)

    return df

In [ ]:
from src.data import prepare_pt_panel_sample, load_pt_panel_data
pt_sample = prepare_pt_panel_sample(load_pt_panel_data())
print(f"PT sample: {pt_sample.shape}")
print(f"Office activations: {pt_sample['year_start'].notna().sum()} municipalities")
print(f"Mean had_office: {pt_sample['had_office'].mean():.2%}")
print(f"Mean I_cons: {pt_sample['I_cons'].mean():.2%}")
display(pt_sample[['cod.1970', 'year', 'new_members', 'year_start', 'had_office', 'I_cons', 'cons']].head(10))


## Data Loading and Validation


### Data Loading and Validation

Load 7 datasets (municipality, diocese, electoral, parish, PT party, bishop classification). Validate completeness and merge keys.

In [ ]:
def load_main_data() -> pd.DataFrame:
    """Load the main analysis dataset from converted data.

    Returns:
        DataFrame with all variables needed for analysis.
    """
    # Load primary datasets
    munidata = pd.read_parquet(DATA_CONVERTED / DATA_FILES["munidata"])
    electoral = pd.read_parquet(DATA_CONVERTED / DATA_FILES["electoral"])

    # Merge municipality data with electoral data
    df = munidata.merge(electoral, on="cod.2010", how="left")

    # Return merged dataset
    return df

def load_diocese_data() -> pd.DataFrame:
    """Load diocese-level data.

    Returns:
        DataFrame with diocese characteristics.
    """
    return pd.read_parquet(DATA_CONVERTED / DATA_FILES["diodata"])

def load_parish_data() -> pd.DataFrame:
    """Load parish-level data for Table 2.

    Returns:
        DataFrame with parish turnover data.
    """
    parish = pd.read_parquet(DATA_CONVERTED / DATA_FILES["parish"])
    diodata = load_diocese_data()

    # Merge with diocese data
    df = parish.merge(diodata, on="CE_1978", how="left")

    return df

def load_pt_panel_data() -> pd.DataFrame:
    """Load PT panel data for Table 3.

    Returns:
        DataFrame with PT affiliation panel.
    """
    pt_panel = pd.read_parquet(DATA_CONVERTED / DATA_FILES["pt_panel"])
    diodata = load_diocese_data()

    # Merge with diocese data
    df = pt_panel.merge(diodata, on="CE_1978", how="left")

    return df

def prepare_analysis_sample(
    df: pd.DataFrame, exclude_archdioceses: bool = True, election_years: list = None
) -> pd.DataFrame:
    """Apply sample restrictions and construct derived variables.

    Args:
        df: Raw loaded DataFrame from load_main_data().
        exclude_archdioceses: If True, exclude archdioceses (CE_TYPE != "A").
        election_years: List of election years to include.

    Returns:
        Analysis-ready DataFrame with all filters and transformations applied.
    """
    df = df.copy()

    # Filter to election years if specified
    if election_years is not None:
        df = df[df["election"].isin(election_years)]

    # Filter to PT presidential elections
    df = df[(df["partido"] == "PT") & (df["cargo"] == "Presidente")]

    # Exclude archdioceses if requested
    if exclude_archdioceses:
        df = df[df["CE_TYPE"] != "A"]

    # Construct derived variables
    df = construct_treatment_variables(df)

    return df

def construct_treatment_variables(df: pd.DataFrame) -> pd.DataFrame:
    """Construct treatment and instrument variables.

    Args:
        df: DataFrame with raw variables.

    Returns:
        DataFrame with derived treatment variables.
    """
    df = df.copy()

    # Retirement_Years (instrument): years since 1978, censored at election
    df["Retirement_Years"] = df.apply(
        lambda row: (
            row["I_YEAR"] - 1978 if row["I_YEAR"] < row["election"] else row["election"] - 1978
        ),
        axis=1,
    )

    # Exposure (treatment): years of JPII bishop exposure, censored at election
    df["Exposure"] = df.apply(
        lambda row: (
            row["JPIIAPT_YEAR"] - 1978
            if row["JPIIAPT_YEAR"] < row["election"]
            else row["election"] - 1978
        ),
        axis=1,
    )

    # Alternative treatment for Table C8
    df["Exposure_alt"] = df.apply(
        lambda row: (
            row["JPIIELEV_YEAR"] - 1978
            if row["JPIIELEV_YEAR"] < row["election"]
            else row["election"] - 1978
        ),
        axis=1,
    )

    return df

def prepare_parish_sample(df: pd.DataFrame) -> pd.DataFrame:
    """Prepare parish turnover sample for Table 2.

    Args:
        df: Raw parish data from load_parish_data().

    Returns:
        Analysis-ready parish sample.
    """
    df = df.copy()

    # Balanced parish panel: parishes exist in all three years
    df = df[(df["parishes_1965"] > 0) & (df["parishes_1970"] > 0) & (df["parishes_1977"] > 0)]

    # Exclude archdioceses
    df = df[df["CE_TYPE"] != "A"]

    # Construct treatment variables
    df["I_cons"] = (df["AC_year"] > df["I_YEAR"]).astype(int)
    df["cons"] = (df["AC_year"] > df["JPIIAPT_YEAR"]).astype(int)

    return df

def prepare_pt_panel_sample(df: pd.DataFrame) -> pd.DataFrame:
    """Prepare PT panel sample for Table 3.

    Args:
        df: Raw PT panel data from load_pt_panel_data().

    Returns:
        Analysis-ready PT panel sample.
    """
    df = df.copy()

    # Exclude archdioceses and rows with missing diocese linkage
    df = df[df["CE_1978"].notna() & (df["CE_TYPE"] != "A")]

    # Construct treatment variables
    df["I_cons"] = (df["year"] > df["I_YEAR"]).astype(int)
    df["cons"] = (df["year"] > df["JPIIAPT_YEAR"]).astype(int)

    # Compute PT office activation threshold
    # year_start = first year with >= 5 new members
    df["year_start"] = df.groupby("cod.1970")["year"].transform(
        lambda years: (
            years[df.loc[years.index, "new_members"] >= 5].min()
            if (df.loc[years.index, "new_members"] >= 5).any()
            else pd.NA
        )
    )

    # had_office: indicator for years after office activation
    # Create mask where year_start is not null and year > year_start
    # Use pd.NA-safe comparison
    has_start = df["year_start"].notna()
    is_after_start = df["year"] > df["year_start"].where(has_start, df["year"].max() + 1)
    df["had_office"] = (has_start & is_after_start).astype(int)

    return df

def get_sample_stats(sample: pd.DataFrame) -> dict:
    """Compute summary statistics for the analysis sample.

    Args:
        sample: Prepared analysis sample from prepare_analysis_sample().

    Returns:
        Dictionary with keys like n_obs, n_units, year_range, etc.
    """
    stats = {
        "n_obs": len(sample),
        "n_dioceses": sample["CE_1978"].nunique() if "CE_1978" in sample.columns else None,
        "n_municipalities": (
            sample["cod.2010"].nunique() if "cod.2010" in sample.columns else None
        ),
        "n_states": sample["UF"].nunique() if "UF" in sample.columns else None,
        "election_years": (
            sorted(sample["election"].unique().tolist()) if "election" in sample.columns else None
        ),
        "mean_vote_share": (
            sample["vote.sh"].mean() if "vote.sh" in sample.columns else None
        ),
        "mean_exposure": sample["Exposure"].mean() if "Exposure" in sample.columns else None,
        "mean_retirement_years": (
            sample["Retirement_Years"].mean() if "Retirement_Years" in sample.columns else None
        ),
    }

    return stats

In [ ]:
# load_main_data() available
# load_diocese_data() available
# load_parish_data() available
# load_pt_panel_data() available
# prepare_analysis_sample() available
# construct_treatment_variables() available
# prepare_parish_sample() available
# prepare_pt_panel_sample() available
# get_sample_stats() available

## Descriptive Analysis


### Descriptive Analysis

Generate maps showing PT territorial expansion (1989-2002) and mandated exposure to progressive bishops. Summary statistics table.

In [ ]:
def run(sample: pd.DataFrame = None):
    """Run all descriptive analyses."""
    print("Running Figure 1 (PT Vote Share Maps)...")
    figure1(sample)

    print("Running Figure 3 (Mandated Exposure Maps)...")
    figure3(sample)

    print("Running Table A1 (Summary Statistics)...")
    tableA1(sample)

    print("Descriptive analyses complete.")

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Tests of Identification Design


### Tests of Identification Design

First stage visualization: scatter plot of mandated vs observed retirement, coefficient over time, F-statistics. Balance tests and placebo regression (1976 MDB vote).

In [ ]:
def run(sample: pd.DataFrame = None):
    """Run all tests of design."""
    print("Running Figure 2 (First-Stage Analysis)...")
    figure2(sample)

    print("Running Table B1 (Balance Tests)...")
    tableB1(sample)

    print("Running Table B2 (Placebo Test)...")
    tableB2(sample)

    print("Running Figure B2 (First-Stage by Exit Type)...")
    figureB2(sample)

    print("Running Figure B3 (Bishop Age Distribution)...")
    figureB3(sample)

    print("Tests of design complete.")

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Main Electoral Analysis (Table 1)


### Main Electoral Analysis (Table 1)

2SLS and reduced-form estimates of progressive bishop exposure on PT presidential vote share. Four elections (1989, 1994, 1998, 2002), state FE, diocese clustering.

In [ ]:
def run(sample: pd.DataFrame = None):
    """Run all electoral analyses."""
    print("Running Table 1 (Main Results)...")
    table1(sample)

    print("Running Table C1 (Diocese-Level)...")
    tableC1(sample)

    print("Running Table C2 (Include Archdioceses)...")
    tableC2(sample)

    print("Running Table C5 (Truncated Sample)...")
    tableC5(sample)

    print("Running Table D1 (All Left Parties)...")
    tableD1(sample)

    print("Electoral analysis complete.")

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Electoral Robustness Checks


### Electoral Robustness Checks

Diocese-level aggregation (C1), archdioceses included (C2), congressional elections (C3), bishop controls (C4), truncated sample (C5), categorical treatment (C6), drop conservatives (C7), alternative timing (C8), all left parties (D1).

In [ ]:
def run(sample: pd.DataFrame = None):
    """Run all electoral analyses."""
    print("Running Table 1 (Main Results)...")
    table1(sample)

    print("Running Table C1 (Diocese-Level)...")
    tableC1(sample)

    print("Running Table C2 (Include Archdioceses)...")
    tableC2(sample)

    print("Running Table C5 (Truncated Sample)...")
    tableC5(sample)

    print("Running Table D1 (All Left Parties)...")
    tableD1(sample)

    print("Electoral analysis complete.")

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Mechanism 1 - Priest Turnover (Table 2)


### Mechanism 1 - Priest Turnover (Table 2)

Test whether progressive bishops increased priest turnover at the parish level. Six specifications: baseline, single-parish munis, secular parishes only, RS state progressives.

In [ ]:
def run(sample: pd.DataFrame = None):
    """Run parish data analysis."""
    print("Running Table 2 (Priest Turnover)...")
    table2(sample)
    print("Parish data analysis complete.")

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Mechanism 2 - PT Local Presence (Table 3)


### Mechanism 2 - PT Local Presence (Table 3)

Test whether progressive bishops increased PT local party presence. Outcomes: office activation year, new members. Panel data 1970-2002.

In [ ]:
def run(sample: pd.DataFrame = None):
    """Run PT local presence analysis."""
    print("Running Table 3 (PT Local Presence)...")
    table3(sample)
    print("PT local presence analysis complete.")

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))